# Kaggle 02 — W3 Synthetic SFT Data Generation

**CPU only** — niente GPU necessaria per questi step.

Step eseguiti qui:
1. **MainframeBench** (~7k esempi curati, MIT) — import diretto da HF, ~5 min
2. **Alibaba gold** (~4k esempi) — DashScope API gratis, cascata qwen3-coder-plus → 235b → max → qwq
   → ~1000 esempi per modello, push ogni 150 esempi, resume automatico
3. ~~Gemini~~ — free tier 20 RPD inutilizzabile; modelli alternativi più deboli del student

Step NON eseguiti qui (richiedono GPU / Modal):
4. Teacher vLLM bulk (~10k) — Modal A100 con XMAiNframe-10.5b

**Prerequisiti Kaggle Secrets:** `HF_TOKEN`, `GH_TOKEN`, `ALIBABA_API`

Output: HF Hub private `AlexThunder0/cobol-sft-dataset`

In [ ]:
!pip install datasets huggingface-hub openai tqdm -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['ALIBABA_API'] = secrets.get_secret('ALIBABA_API')

print('Secrets configurati (HF + Alibaba)')

In [ ]:
import subprocess, sys, os, shutil

GH_TOKEN = secrets.get_secret('GH_TOKEN')
shutil.rmtree('/kaggle/working/Qwen-COBOL', ignore_errors=True)

with open(os.path.expanduser('~/.netrc'), 'w') as f:
    f.write(f'machine github.com login x-access-token password {GH_TOKEN}\n')
os.chmod(os.path.expanduser('~/.netrc'), 0o600)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--quiet',
     'https://github.com/AlexThunder01/Qwen-COBOL.git',
     '/kaggle/working/Qwen-COBOL'],
    check=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
sys.path.insert(0, '/kaggle/working/Qwen-COBOL')
print('Repo clonato')

In [ ]:
# ── Step 1: MainframeBench (~7k esempi, MIT, CPU, ~5 min) ────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

from src.synth.distill_orchestrator import step_mainframebench
step_mainframebench()
print('MainframeBench importato.')

In [ ]:
# ── Step 2: Alibaba gold (~4k esempi, CPU, resume automatico) ────────────────
# Cascata: qwen3-coder-plus → qwen3-235b-a22b-thinking-2507 → qwen3-max → qwq-plus
# ~1000 esempi per modello, push ogni 150 esempi (~15 min).
# Può girare per ore senza supervisione — riprende da HF Hub se interrotto.

from src.synth.distill_orchestrator import step_alibaba
step_alibaba()
print('Alibaba gold done.')

In [ ]:
# ── Statistiche dataset SFT corrente ─────────────────────────────────────────
from datasets import load_dataset
import pandas as pd

splits_info = []
for split in ['mainframebench', 'teacher_bulk', 'alibaba_gold']:
    try:
        ds = load_dataset('AlexThunder0/cobol-sft-dataset', split=split,
                         token=os.environ['HF_TOKEN'])
        splits_info.append({'split': split, 'esempi': len(ds)})
    except Exception as e:
        splits_info.append({'split': split, 'esempi': f'non trovato'})

df = pd.DataFrame(splits_info)
total = sum(r['esempi'] for r in splits_info if isinstance(r['esempi'], int))
print('=== SFT DATASET STATS ===')
print(df.to_string(index=False))
print(f'\nTotale: {total:,} esempi')